# Phase 3 Part 2: SGL on All 5 Datasets
**CSC14114 - Big Data Applications | Official SGL-Torch workflow**

This notebook is **Phase 3 - Part 2** only.

## Phase 3 split
1. Part 1: run **LightGCN** on all 5 datasets
2. Part 2: run **SGL** on all 5 datasets
3. Part 3: run **SimGCL** on all 5 datasets and compare all 3 models

## What this notebook does
- uses the official PyTorch SGL repository: `wujcan/SGL-Torch`
- keeps the same 5 datasets and split-building logic as Part 1
- runs **3 repeated runs with fixed seed 42** per dataset
- uses the official SGL training loop and built-in early stopping (`stop_cnt`)
- reports `Recall@20`, `NDCG@20`, `HR@20`, `MRR@20`
- saves per-run logs, best checkpoints, averaged results, runtime, and convergence curves

## Datasets used in Part 2
- `ml-1m`
- `gowalla`
- `yelp2018`
- `amazon-book`
- `lastfm_phase2`

## Official references
- Repo: `https://github.com/wujcan/SGL-Torch`
- Paper: `https://arxiv.org/abs/2010.10783`

## Kaggle setup
- Turn **Internet = ON**
- Turn **GPU = ON**
- Upload this notebook
- Add a Kaggle input dataset that contains either:
  - `lastfm_phase2_processed.zip`, or
  - extracted files `interactions.csv`, `users.csv`, `items.csv`, `manifest.json`
- Then click **Run All** and **Save Version**


In [ ]:
import json
import os
import random
import re
import shutil
import subprocess
import sys
import time
import zipfile
from pathlib import Path

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
import torch

ROOT = Path('/kaggle/working/phase3_part2_sgl') if Path('/kaggle').exists() else Path('phase3_part2_sgl')
WORK = ROOT / 'workspace'
REPO_DIR = WORK / 'SGL-Torch'
RESULTS_DIR = ROOT / 'results'
FIG_DIR = RESULTS_DIR / 'figures'
AVG_RESULTS_DIR = RESULTS_DIR / 'average_results_3_runs'
INPUT_DIR = Path('/kaggle/input') if Path('/kaggle/input').exists() else Path.cwd()
for p in [WORK, RESULTS_DIR, FIG_DIR, AVG_RESULTS_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print('Workspace:', ROOT.resolve())
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))


In [ ]:
LASTFM_REQUIRED_FILES = ['interactions.csv', 'users.csv', 'items.csv', 'manifest.json']


def find_lastfm_phase2_source(input_dir: Path):
    zip_candidates = sorted(input_dir.rglob('*.zip'))
    for zip_path in zip_candidates:
        try:
            with zipfile.ZipFile(zip_path) as zf:
                names = {Path(name).name for name in zf.namelist() if not name.endswith('/')}
            if all(name in names for name in LASTFM_REQUIRED_FILES):
                return ('zip', zip_path)
        except zipfile.BadZipFile:
            continue

    seen = set()
    for manifest_path in sorted(input_dir.rglob('manifest.json')):
        folder = manifest_path.parent
        folder_key = str(folder.resolve())
        if folder_key in seen:
            continue
        seen.add(folder_key)
        if all((folder / name).exists() for name in LASTFM_REQUIRED_FILES):
            return ('dir', folder)

    raise FileNotFoundError(
        'Could not find Last.fm Phase 2 input data under /kaggle/input. '
        'Attach a Kaggle dataset containing lastfm_phase2_processed.zip or the extracted processed CSV/JSON files.'
    )


def materialize_lastfm_phase2_input() -> Path:
    target_dir = WORK / 'lastfm_phase2_input'
    if target_dir.exists() and all((target_dir / name).exists() for name in LASTFM_REQUIRED_FILES):
        return target_dir

    if target_dir.exists():
        shutil.rmtree(target_dir)
    target_dir.mkdir(parents=True, exist_ok=True)

    source_kind, source_path = find_lastfm_phase2_source(INPUT_DIR)
    if source_kind == 'zip':
        with zipfile.ZipFile(source_path) as zf:
            zf.extractall(target_dir)
    else:
        for name in LASTFM_REQUIRED_FILES:
            shutil.copy2(source_path / name, target_dir / name)

    return target_dir


LASTFM_PHASE2_INPUT_DIR = materialize_lastfm_phase2_input()
print(f'Last.fm Phase 2 input ready at: {LASTFM_PHASE2_INPUT_DIR}')


In [ ]:
REPO_URL = 'https://github.com/wujcan/SGL-Torch.git'
#DATASETS = ['ml-1m', 'gowalla', 'yelp2018', 'amazon-book', 'lastfm_phase2']
DATASETS = ['ml-1m']
FIXED_SEED = 42
RUN_IDS = [1, 2, 3]
TOPKS = [20]
REPORT_TOPK = 20
METRIC_LIST = ['Precision', 'Recall', 'NDCG', 'MRR']

LIGHTGCN_RAW_SPLIT_URLS = {
    'gowalla': {
        'train': 'https://raw.githubusercontent.com/gusye1234/LightGCN-PyTorch/master/data/gowalla/train.txt',
        'test': 'https://raw.githubusercontent.com/gusye1234/LightGCN-PyTorch/master/data/gowalla/test.txt',
    },
    'yelp2018': {
        'train': 'https://raw.githubusercontent.com/gusye1234/LightGCN-PyTorch/master/data/yelp2018/train.txt',
        'test': 'https://raw.githubusercontent.com/gusye1234/LightGCN-PyTorch/master/data/yelp2018/test.txt',
    },
    'amazon-book': {
        'train': 'https://raw.githubusercontent.com/gusye1234/LightGCN-PyTorch/master/data/amazon-book/train.txt',
        'test': 'https://raw.githubusercontent.com/gusye1234/LightGCN-PyTorch/master/data/amazon-book/test.txt',
    },
}

MOVIELENS_URL = 'https://files.grouplens.org/datasets/movielens/ml-1m.zip'

BASE_SGL_CONFIG = {
    'aug_type': 'ED',
    'reg': 1e-4,
    'embed_size': 64,
    'n_layers': 3,
    'ssl_reg': 0.1,
    'ssl_ratio': 0.1,
    'ssl_temp': 0.2,
    'ssl_mode': 'both_side',
    'lr': 0.001,
    'learner': 'adam',
    'adj_type': 'pre',
    'epochs': 200,
    'batch_size': 2048,
    'num_negatives': 1,
    'param_init': 'xavier_uniform',
    'stddev': 0.01,
    'verbose': 1,
    'stop_cnt': 10,
    'test_thread': 8,
    'test_batch_size': 128,
    'gpu_id': 0,
}

SGL_RUN_CONFIG = {
    'ml-1m': dict(BASE_SGL_CONFIG),
    'gowalla': dict(BASE_SGL_CONFIG),
    'yelp2018': dict(BASE_SGL_CONFIG),
    'amazon-book': dict(BASE_SGL_CONFIG),
    'lastfm_phase2': dict(BASE_SGL_CONFIG),
}
SGL_RUN_CONFIG['amazon-book']['ssl_reg'] = 0.5


In [ ]:
def run_cmd(cmd, cwd=None, capture_output=False, env=None):
    print('$', ' '.join(str(x) for x in cmd))
    return subprocess.run(cmd, cwd=cwd, check=True, text=True, capture_output=capture_output, env=env)


def download_file(url: str, dest: Path):
    dest.parent.mkdir(parents=True, exist_ok=True)
    if dest.exists():
        print(f'[skip] {dest.name}')
        return dest
    print(f'[download] {url}')
    response = requests.get(url, stream=True, timeout=300)
    response.raise_for_status()
    with dest.open('wb') as f:
        for chunk in response.iter_content(1024 * 1024):
            if chunk:
                f.write(chunk)
    return dest


def build_leave_one_out_maps(df: pd.DataFrame, user_col: str, item_col: str, time_col: str):
    work = df[[user_col, item_col, time_col]].copy()
    work = work.sort_values([user_col, time_col, item_col], kind='mergesort')
    users = sorted(work[user_col].unique().tolist())
    items = sorted(work[item_col].unique().tolist())
    u_map = {u: i for i, u in enumerate(users)}
    i_map = {it: i for i, it in enumerate(items)}
    work['uid'] = work[user_col].map(u_map).astype(int)
    work['iid'] = work[item_col].map(i_map).astype(int)

    train_map = {}
    test_map = {}
    for uid, group in work.groupby('uid', sort=True):
        item_list = group['iid'].tolist()
        if len(item_list) < 2:
            continue
        train_map[int(uid)] = [int(x) for x in item_list[:-1]]
        test_map[int(uid)] = [int(item_list[-1])]
    return train_map, test_map


def maps_to_pairs(data_map: dict):
    pairs = []
    for uid in sorted(data_map):
        for iid in data_map[uid]:
            pairs.append((int(uid), int(iid)))
    return pairs


def parse_lightgcn_split(path: Path):
    pairs = []
    with path.open('r', encoding='utf-8') as f:
        for line in f:
            parts = line.strip().split()
            if not parts:
                continue
            uid = int(parts[0])
            item_list = [int(x) for x in parts[1:]]
            for iid in item_list:
                pairs.append((uid, iid))
    return pairs


def write_sgl_pairs(folder: Path, dataset_name: str, train_pairs, test_pairs):
    folder.mkdir(parents=True, exist_ok=True)
    train_df = pd.DataFrame(train_pairs, columns=['user', 'item']).sort_values(['user', 'item']).reset_index(drop=True)
    test_df = pd.DataFrame(test_pairs, columns=['user', 'item']).sort_values(['user', 'item']).reset_index(drop=True)
    train_df.to_csv(folder / f'{dataset_name}.train', index=False, header=False)
    test_df.to_csv(folder / f'{dataset_name}.test', index=False, header=False)


def clone_and_patch_repo():
    if not REPO_DIR.exists():
        run_cmd(['git', 'clone', REPO_URL, str(REPO_DIR)])

    try:
        import Cython  # noqa: F401
    except ImportError:
        try:
            run_cmd([sys.executable, '-m', 'pip', 'install', '-q', 'cython'], cwd=REPO_DIR)
        except subprocess.CalledProcessError as exc:
            print(f'[warning] Cython install failed; compile step may fall back to pure Python. {exc}')

    compatibility_targets = [
        REPO_DIR / 'data' / 'sampler.py',
        REPO_DIR / 'reckit' / 'util' / 'decorators.py',
    ]
    for compat_path in compatibility_targets:
        compat_text = compat_path.read_text(encoding='utf-8')
        updated_text = compat_text.replace(
            'from collections import Iterable',
            'from collections.abc import Iterable',
        )
        if updated_text != compat_text:
            compat_path.write_text(updated_text, encoding='utf-8')

    main_phase3_script = '''
from importlib import import_module
from importlib.util import find_spec
from pathlib import Path

import numpy as np
import os
import random
import torch

from reckit import Configurator
from reckit import typeassert


def _set_random_seed(seed=2020):
    np.random.seed(seed)
    random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    print('set pytorch seed')


@typeassert(recommender=str)
def find_recommender(recommender):
    model_dirs = set(os.listdir('model'))
    model_dirs.remove('base')

    module = None
    for tdir in model_dirs:
        spec_path = '.'.join(['model', tdir, recommender])
        if find_spec(spec_path):
            module = import_module(spec_path)
            break

    if module is None:
        raise ImportError(f'Recommender: {recommender} not found')

    if hasattr(module, recommender):
        return getattr(module, recommender)
    raise ImportError(f'Import {recommender} failed from {module.__file__}!')


if __name__ == '__main__':
    repo_root = Path(__file__).resolve().parent
    root_dir = repo_root.as_posix() + '/'
    data_dir = (repo_root / 'dataset').as_posix() + '/'

    config = Configurator(root_dir, data_dir)
    config.add_config(str(repo_root / 'NeuRec.ini'), section='NeuRec')
    config.parse_cmd()
    os.environ['CUDA_VISIBLE_DEVICES'] = str(config['gpu_id'])
    _set_random_seed(config['seed'])
    Recommender = find_recommender(config.recommender)

    model_cfg = str(repo_root / 'conf' / f'{config.recommender}.ini')
    config.add_config(model_cfg, section='hyperparameters', used_as_summary=True)

    recommender = Recommender(config)
    recommender.train_model()
'''.lstrip()
    (REPO_DIR / 'main_phase3.py').write_text(main_phase3_script, encoding='utf-8')

    metric_eval_script = '''
import json
import os
from pathlib import Path

import numpy as np
import torch

from reckit import Configurator
from main_phase3 import _set_random_seed, find_recommender


def user_metrics(top_items, truth_items, k):
    hits = [1.0 if item in truth_items else 0.0 for item in top_items[:k]]
    precision = sum(hits) / k
    recall = sum(hits) / len(truth_items) if truth_items else 0.0
    hr = 1.0 if any(hits) else 0.0

    dcg = 0.0
    for idx, hit in enumerate(hits):
        if hit:
            dcg += 1.0 / np.log2(idx + 2.0)
    ideal_len = min(len(truth_items), k)
    idcg = sum(1.0 / np.log2(idx + 2.0) for idx in range(ideal_len))
    ndcg = dcg / idcg if idcg > 0 else 0.0

    mrr = 0.0
    for idx, hit in enumerate(hits):
        if hit:
            mrr = 1.0 / (idx + 1.0)
            break
    return precision, recall, ndcg, hr, mrr


def build_config():
    repo_root = Path(__file__).resolve().parent
    root_dir = repo_root.as_posix() + '/'
    data_dir = (repo_root / 'dataset').as_posix() + '/'
    config = Configurator(root_dir, data_dir)
    config.add_config(str(repo_root / 'NeuRec.ini'), section='NeuRec')
    config.parse_cmd()
    model_cfg = str(repo_root / 'conf' / f'{config.recommender}.ini')
    config.add_config(model_cfg, section='hyperparameters', used_as_summary=True)
    return config


def evaluate_ranking(recommender, topks):
    user_train_dict = recommender.dataset.train_data.to_user_dict()
    user_test_dict = recommender.dataset.test_data.to_user_dict()
    users = list(user_test_dict.keys())
    max_k = max(topks)
    totals = {}
    for k in topks:
        totals[f'precision@{k}'] = 0.0
        totals[f'recall@{k}'] = 0.0
        totals[f'ndcg@{k}'] = 0.0
        totals[f'hr@{k}'] = 0.0
        totals[f'mrr@{k}'] = 0.0

    batch_size = int(recommender.test_batch_size)
    for start in range(0, len(users), batch_size):
        batch_users = users[start:start + batch_size]
        ranking_score = recommender.predict(batch_users)
        if not isinstance(ranking_score, np.ndarray):
            ranking_score = np.asarray(ranking_score, dtype=np.float32)

        for idx, user in enumerate(batch_users):
            train_items = user_train_dict.get(user)
            if train_items is not None and len(train_items) > 0:
                ranking_score[idx][train_items] = -np.inf

        top_part = np.argpartition(-ranking_score, kth=max_k - 1, axis=1)[:, :max_k]
        top_scores = np.take_along_axis(ranking_score, top_part, axis=1)
        top_order = np.argsort(-top_scores, axis=1)
        top_items = np.take_along_axis(top_part, top_order, axis=1)

        for row_idx, user in enumerate(batch_users):
            truth_items = set(np.asarray(user_test_dict[user]).tolist())
            ranked_items = top_items[row_idx].tolist()
            for k in topks:
                precision, recall, ndcg, hr, mrr = user_metrics(ranked_items, truth_items, k)
                totals[f'precision@{k}'] += precision
                totals[f'recall@{k}'] += recall
                totals[f'ndcg@{k}'] += ndcg
                totals[f'hr@{k}'] += hr
                totals[f'mrr@{k}'] += mrr

    user_count = max(len(users), 1)
    return {key: value / user_count for key, value in totals.items()}


if __name__ == '__main__':
    config = build_config()
    os.environ['CUDA_VISIBLE_DEVICES'] = str(config['gpu_id'])
    _set_random_seed(config['seed'])
    Recommender = find_recommender(config.recommender)
    recommender = Recommender(config)

    checkpoint_path = Path(config['checkpoint'])
    output_path = Path(config['output'])
    state_dict = torch.load(checkpoint_path, map_location=recommender.device)
    recommender.lightgcn.load_state_dict(state_dict)

    metrics = evaluate_ranking(recommender, list(config.top_k))
    output_path.write_text(json.dumps(metrics, indent=2), encoding='utf-8')
    print(json.dumps(metrics, indent=2))
'''.lstrip()
    (REPO_DIR / 'metric_eval.py').write_text(metric_eval_script, encoding='utf-8')

    cython_init_script = '''
import numpy as np

float_type = np.float32
int_type = np.int32


def is_ndarray(array, dtype=None):
    if not isinstance(array, np.ndarray):
        return False
    return dtype is None or array.dtype == dtype


def _prepare_candidates(scores, top_k):
    k = min(top_k, scores.shape[0])
    if k <= 0:
        return [-1] * top_k
    top_idx = np.argpartition(-scores, kth=k - 1)[:k]
    top_scores = scores[top_idx]
    top_order = np.argsort(-top_scores)
    ranked = top_idx[top_order].tolist()
    if len(ranked) < top_k:
        ranked.extend([-1] * (top_k - len(ranked)))
    return ranked


def _metric_curves(ranked_items, truth_items, top_k):
    truth_set = set(int(x) for x in truth_items)
    truth_len = len(truth_set)
    curves = {1: [], 2: [], 3: [], 4: [], 5: []}
    hits = 0.0
    ap_sum = 0.0
    dcg = 0.0
    first_rr = 0.0

    for rank, item in enumerate(ranked_items[:top_k], start=1):
        hit = item in truth_set
        if hit:
            hits += 1.0
            ap_sum += hits / rank
            dcg += 1.0 / np.log2(rank + 1.0)
            if first_rr == 0.0:
                first_rr = 1.0 / rank

        ideal_hits = min(truth_len, rank)
        idcg = sum(1.0 / np.log2(idx + 2.0) for idx in range(ideal_hits))

        curves[1].append(hits / rank)
        curves[2].append(hits / truth_len if truth_len else 0.0)
        curves[3].append(ap_sum / truth_len if truth_len else 0.0)
        curves[4].append(dcg / idcg if idcg > 0 else 0.0)
        curves[5].append(first_rr)

    return curves


def pyx_randint_choice(high, size=1, replace=True, p=None, exclusion=None):
    candidates = np.arange(high)
    if exclusion is not None:
        exclusion = np.asarray(exclusion, dtype=np.int64)
        candidates = candidates[~np.isin(candidates, exclusion)]
    if candidates.size == 0:
        raise ValueError('No candidates left for randint_choice fallback.')

    probs = None
    if p is not None:
        probs = np.asarray(p, dtype=np.float64)
        if probs.shape[0] != high:
            raise ValueError('Probability vector length does not match high.')
        probs = probs[candidates]
        total = probs.sum()
        probs = None if total <= 0 else probs / total

    sampled = np.random.choice(candidates, size=int(size), replace=replace, p=probs)
    return np.asarray(sampled, dtype=np.int64)


def pyx_batch_randint_choice(high, size, replace=True, p=None, exclusion=None, thread_num=1):
    size_arr = np.asarray(size, dtype=np.int64).reshape(-1)
    results = []
    for idx, sample_size in enumerate(size_arr.tolist()):
        probs = None if p is None else p[idx]
        blocked = None if exclusion is None else exclusion[idx]
        results.append(pyx_randint_choice(high, size=sample_size, replace=replace, p=probs, exclusion=blocked))
    return results


def eval_score_matrix(score_matrix, test_items, metric, top_k, thread_num):
    scores = np.asarray(score_matrix, dtype=float_type)
    metric_ids = [int(m) for m in metric]
    results = np.zeros((scores.shape[0], len(metric_ids) * top_k), dtype=float_type)

    for row_idx in range(scores.shape[0]):
        ranked_items = _prepare_candidates(scores[row_idx], top_k)
        curves = _metric_curves(ranked_items, test_items[row_idx], top_k)
        for metric_pos, metric_id in enumerate(metric_ids):
            start = metric_pos * top_k
            results[row_idx, start:start + top_k] = np.asarray(curves[metric_id], dtype=float_type)

    return results


try:
    from .eval_matrix import eval_score_matrix as _compiled_eval_score_matrix
    from .random_choice import pyx_randint_choice as _compiled_randint_choice
    from .random_choice import pyx_batch_randint_choice as _compiled_batch_randint_choice
    from .tools import is_ndarray as _compiled_is_ndarray, float_type as _compiled_float_type, int_type as _compiled_int_type
except ImportError:
    pass
else:
    eval_score_matrix = _compiled_eval_score_matrix
    pyx_randint_choice = _compiled_randint_choice
    pyx_batch_randint_choice = _compiled_batch_randint_choice
    is_ndarray = _compiled_is_ndarray
    float_type = _compiled_float_type
    int_type = _compiled_int_type
'''.lstrip()
    (REPO_DIR / 'reckit' / 'cython' / '__init__.py').write_text(cython_init_script, encoding='utf-8')

    sort_script = '''
__author__ = "Zhongchuan Sun"
__email__ = "zhongchuansun@gmail.com"

__all__ = ["sort", "arg_sort", "top_k", "arg_top_k"]

import numpy as np

try:
    from reckit.cython.sort import pyx_sort, pyx_arg_sort, pyx_top_k, pyx_arg_top_k
except ImportError:
    def sort(array, reverse=False, n_threads=1):
        arr = np.asarray(array)
        result = np.sort(arr, axis=-1)
        return result[..., ::-1] if reverse else result

    def arg_sort(array, reverse=False, n_threads=1):
        arr = np.asarray(array)
        result = np.argsort(arr, axis=-1)
        return result[..., ::-1] if reverse else result

    def top_k(array, topk, n_threads=1):
        arr = np.asarray(array)
        return np.take_along_axis(arr, arg_top_k(arr, topk, n_threads=n_threads), axis=-1)

    def arg_top_k(array, topk, n_threads=1):
        arr = np.asarray(array)
        topk = int(topk)
        if arr.ndim == 1:
            k = min(topk, arr.shape[0])
            idx = np.argpartition(-arr, kth=max(k - 1, 0))[:k]
            order = np.argsort(-arr[idx])
            return idx[order]

        k = min(topk, arr.shape[1])
        idx = np.argpartition(-arr, kth=max(k - 1, 0), axis=1)[:, :k]
        scores = np.take_along_axis(arr, idx, axis=1)
        order = np.argsort(-scores, axis=1)
        return np.take_along_axis(idx, order, axis=1)
else:
    def sort(array, reverse=False, n_threads=1):
        return pyx_sort(array, reverse, n_threads)

    def arg_sort(array, reverse=False, n_threads=1):
        return pyx_arg_sort(array, reverse, n_threads)

    def top_k(array, topk, n_threads=1):
        return pyx_top_k(array, topk, n_threads)

    def arg_top_k(array, topk, n_threads=1):
        return pyx_arg_top_k(array, topk, n_threads)
'''.lstrip()
    (REPO_DIR / 'reckit' / 'sort.py').write_text(sort_script, encoding='utf-8')

    sgl_path = REPO_DIR / 'model' / 'general_recommender' / 'SGL.py'
    sgl_text = sgl_path.read_text(encoding='utf-8')
    if 'SGL_RUN_OUTPUT_DIR' not in sgl_text:
        old_import = '''import numpy as np
from time import time
'''
        new_import = '''import numpy as np
import json
import os
from time import time
'''
        if old_import not in sgl_text:
            raise ValueError('Could not patch SGL.py imports.')
        sgl_text = sgl_text.replace(old_import, new_import)

        old_optimizer = '''        self.optimizer = torch.optim.Adam(self.lightgcn.parameters(), lr=self.lr)
'''
        new_optimizer = '''        self.optimizer = torch.optim.Adam(self.lightgcn.parameters(), lr=self.lr)

        self.run_output_dir = os.environ.get('SGL_RUN_OUTPUT_DIR')
        self.best_model_path = None
        self.training_summary_path = None
        if self.run_output_dir:
            os.makedirs(self.run_output_dir, exist_ok=True)
            self.best_model_path = os.path.join(self.run_output_dir, 'best_model.pt')
            self.training_summary_path = os.path.join(self.run_output_dir, 'training_summary.json')
'''
        if old_optimizer not in sgl_text:
            raise ValueError('Could not patch SGL.py optimizer block.')
        sgl_text = sgl_text.replace(old_optimizer, new_optimizer)

        old_stopping = '''        stopping_step = 0
'''
        new_stopping = '''        stopping_step = 0
        stop_epoch = self.epochs
'''
        if old_stopping not in sgl_text:
            raise ValueError('Could not patch SGL.py stopping setup.')
        sgl_text = sgl_text.replace(old_stopping, new_stopping, 1)

        old_flag_block = '''                if flag:
                    self.best_epoch = epoch
                    stopping_step = 0
                    self.logger.info("Find a better model.")
                    if self.save_flag:
                        self.logger.info("Save model to file as pretrain.")
                        torch.save(self.lightgcn.state_dict(), self.tmp_model_dir)
                        self.saver.save(self.sess, self.tmp_model_dir)
'''
        new_flag_block = '''                if flag:
                    self.best_epoch = epoch
                    stopping_step = 0
                    self.logger.info("Find a better model.")
                    if self.best_model_path is not None:
                        torch.save(self.lightgcn.state_dict(), self.best_model_path)
                    if self.save_flag:
                        self.logger.info("Save model to file as pretrain.")
                        torch.save(self.lightgcn.state_dict(), self.tmp_model_dir)
                        self.saver.save(self.sess, self.tmp_model_dir)
'''
        if old_flag_block not in sgl_text:
            raise ValueError('Could not patch SGL.py best-model block.')
        sgl_text = sgl_text.replace(old_flag_block, new_flag_block)

        old_early_stop = '''                else:
                    stopping_step += 1
                    if stopping_step >= self.stop_cnt:
                        self.logger.info("Early stopping is trigger at epoch: {}".format(epoch))
                        break
'''
        new_early_stop = '''                else:
                    stopping_step += 1
                    if stopping_step >= self.stop_cnt:
                        self.logger.info("Early stopping is trigger at epoch: {}".format(epoch))
                        stop_epoch = epoch
                        break
'''
        if old_early_stop not in sgl_text:
            raise ValueError('Could not patch SGL.py early-stop block.')
        sgl_text = sgl_text.replace(old_early_stop, new_early_stop)

        old_final = '''        self.logger.info("\\t\\t%s" % buf)
'''
        new_final = '''        self.logger.info("\\t\\t%s" % buf)
        if self.training_summary_path is not None:
            summary = {
                'best_epoch': int(self.best_epoch),
                'best_result': [float(x) for x in np.asarray(self.best_result).reshape(-1)],
                'stop_epoch': int(stop_epoch),
                'stopped_early': bool(stop_epoch < self.epochs),
            }
            with open(self.training_summary_path, 'w', encoding='utf-8') as f:
                json.dump(summary, f, indent=2)
'''
        if old_final not in sgl_text:
            raise ValueError('Could not patch SGL.py training summary block.')
        sgl_text = sgl_text.replace(old_final, new_final)
        sgl_path.write_text(sgl_text, encoding='utf-8')

    compile_ok = True
    try:
        run_cmd([sys.executable, 'local_compile_setup.py', 'build_ext', '--inplace'], cwd=REPO_DIR)
    except subprocess.CalledProcessError as exc:
        compile_ok = False
        print(f'[warning] Cython compilation failed; using Python fallback path. {exc}')

    if compile_ok:
        print('Official SGL repo is ready with compiled Cython extensions.')
    else:
        print('Official SGL repo is ready with Python fallback evaluator/sampler utilities.')


In [ ]:
def prepare_standard_sgl_datasets():
    raw_root = WORK / 'lightgcn_raw_splits'
    for name, urls in LIGHTGCN_RAW_SPLIT_URLS.items():
        folder = REPO_DIR / 'dataset' / name
        train_file = folder / f'{name}.train'
        test_file = folder / f'{name}.test'
        if train_file.exists() and test_file.exists():
            print(f'{name} already prepared')
            continue
        train_path = download_file(urls['train'], raw_root / name / 'train.txt')
        test_path = download_file(urls['test'], raw_root / name / 'test.txt')
        train_pairs = parse_lightgcn_split(train_path)
        test_pairs = parse_lightgcn_split(test_path)
        write_sgl_pairs(folder, name, train_pairs, test_pairs)
        print(f'Prepared {name}')


def prepare_ml_1m_dataset():
    folder = REPO_DIR / 'dataset' / 'ml-1m'
    train_file = folder / 'ml-1m.train'
    test_file = folder / 'ml-1m.test'
    if train_file.exists() and test_file.exists():
        print('ml-1m already prepared')
        return

    zip_path = download_file(MOVIELENS_URL, WORK / 'ml-1m.zip')
    extract_dir = WORK / 'ml-1m_extracted'
    if not extract_dir.exists():
        with zipfile.ZipFile(zip_path) as zf:
            zf.extractall(extract_dir)

    ratings_path = extract_dir / 'ml-1m' / 'ratings.dat'
    df = pd.read_csv(
        ratings_path,
        sep='::',
        header=None,
        names=['user_id', 'item_id', 'rating', 'timestamp'],
        engine='python',
        encoding='latin-1',
    )
    df = df[['user_id', 'item_id', 'timestamp']].drop_duplicates(['user_id', 'item_id'])
    train_map, test_map = build_leave_one_out_maps(df, 'user_id', 'item_id', 'timestamp')
    write_sgl_pairs(folder, 'ml-1m', maps_to_pairs(train_map), maps_to_pairs(test_map))
    print('Prepared ml-1m')


def prepare_lastfm_phase2_dataset():
    folder = REPO_DIR / 'dataset' / 'lastfm_phase2'
    train_file = folder / 'lastfm_phase2.train'
    test_file = folder / 'lastfm_phase2.test'
    if train_file.exists() and test_file.exists():
        print('lastfm_phase2 already prepared')
        return

    df = pd.read_csv(LASTFM_PHASE2_INPUT_DIR / 'interactions.csv')
    df['timestamp'] = pd.to_datetime(df['crawl_timestamp_utc'], utc=True, errors='coerce')
    df = df.dropna(subset=['timestamp']).copy()
    df['timestamp'] = df['timestamp'].astype('int64') // 10**9
    df = df[['user_id', 'item_id', 'timestamp']].drop_duplicates(['user_id', 'item_id'])
    train_map, test_map = build_leave_one_out_maps(df, 'user_id', 'item_id', 'timestamp')
    write_sgl_pairs(folder, 'lastfm_phase2', maps_to_pairs(train_map), maps_to_pairs(test_map))
    print('Prepared lastfm_phase2')


clone_and_patch_repo()
prepare_standard_sgl_datasets()
prepare_ml_1m_dataset()
prepare_lastfm_phase2_dataset()

print('Prepared datasets:')
for p in sorted((REPO_DIR / 'dataset').iterdir()):
    if p.is_dir():
        print('-', p.name)


In [ ]:
def build_sgl_cli_args(dataset: str, cfg: dict):
    return [
        '--recommender=SGL',
        f'--dataset={dataset}',
        '--file_column=UI',
        '--sep=,',
        f'--metric={json.dumps(METRIC_LIST)}',
        f'--top_k={json.dumps(TOPKS)}',
        f"--test_thread={cfg['test_thread']}",
        f"--test_batch_size={cfg['test_batch_size']}",
        f'--seed={FIXED_SEED}',
        f"--gpu_id={cfg['gpu_id']}",
        '--start_testing_epoch=0',
        f"--aug_type={cfg['aug_type']}",
        f"--reg={cfg['reg']}",
        f"--embed_size={cfg['embed_size']}",
        f"--n_layers={cfg['n_layers']}",
        f"--ssl_reg={cfg['ssl_reg']}",
        f"--ssl_ratio={cfg['ssl_ratio']}",
        f"--ssl_temp={cfg['ssl_temp']}",
        f"--ssl_mode={cfg['ssl_mode']}",
        f"--lr={cfg['lr']}",
        f"--learner={cfg['learner']}",
        f"--adj_type={cfg['adj_type']}",
        f"--epochs={cfg['epochs']}",
        f"--batch_size={cfg['batch_size']}",
        f"--num_negatives={cfg['num_negatives']}",
        f"--param_init={cfg['param_init']}",
        f"--stddev={cfg['stddev']}",
        f"--verbose={cfg['verbose']}",
        f"--stop_cnt={cfg['stop_cnt']}",
        '--pretrain_flag=0',
        '--save_flag=0',
    ]


def parse_metric_history(log_text: str):
    metric_names = []
    rows = []
    best_epoch = None
    stop_epoch = None

    for line in log_text.splitlines():
        if 'metrics:' in line:
            metric_names = [name.lower() for name in re.findall(r'[A-Za-z]+@\d+', line)]
            continue

        match = re.search(r'epoch\s+(\d+):\s*(.*)$', line)
        if match and metric_names:
            epoch = int(match.group(1))
            values = [float(x) for x in re.findall(r'-?\d+\.\d+', match.group(2))]
            if len(values) >= len(metric_names):
                row = {'eval_epoch': epoch}
                for name, value in zip(metric_names, values[:len(metric_names)]):
                    row[name] = value
                rows.append(row)
            continue

        match = re.search(r'best_result@epoch\s+(\d+)', line)
        if match:
            best_epoch = int(match.group(1))
            continue

        match = re.search(r'Early stopping is trigger at epoch:\s*(\d+)', line)
        if match:
            stop_epoch = int(match.group(1))

    return rows, metric_names, best_epoch, stop_epoch


def find_repo_log(log_dir: Path, before_logs, start_time: float):
    if not log_dir.exists():
        raise FileNotFoundError(f'Log directory not found: {log_dir}')
    after_logs = {p for p in log_dir.glob('*.log') if p.is_file()}
    new_logs = list(after_logs - before_logs)
    if new_logs:
        return max(new_logs, key=lambda p: p.stat().st_mtime)
    recent_logs = [p for p in after_logs if p.stat().st_mtime >= start_time - 1]
    if recent_logs:
        return max(recent_logs, key=lambda p: p.stat().st_mtime)
    if after_logs:
        return max(after_logs, key=lambda p: p.stat().st_mtime)
    raise FileNotFoundError(f'No SGL log file found in {log_dir}')


def evaluate_extra_metrics(dataset: str, cfg: dict, run_dir: Path):
    checkpoint_path = (run_dir / 'best_model.pt').resolve()
    if not checkpoint_path.exists():
        raise FileNotFoundError(f'Best checkpoint not found: {checkpoint_path}')

    output_path = (run_dir / 'ranking_metrics.json').resolve()
    cmd = [
        sys.executable, 'metric_eval.py',
        *build_sgl_cli_args(dataset, cfg),
        f'--checkpoint={checkpoint_path.as_posix()}',
        f'--output={output_path.as_posix()}',
    ]
    proc = subprocess.run(cmd, cwd=REPO_DIR, text=True, capture_output=True)
    (run_dir / 'metric_eval.log').write_text(proc.stdout + '\n' + proc.stderr, encoding='utf-8')
    if proc.returncode != 0:
        raise RuntimeError(f'Extra metric evaluation failed for dataset={dataset}. Check {run_dir / "metric_eval.log"}.')
    return json.loads(output_path.read_text(encoding='utf-8'))


def run_one(dataset: str, run_id: int):
    cfg = SGL_RUN_CONFIG[dataset]
    run_name = f'run_{run_id}'
    run_dir = RESULTS_DIR / dataset / run_name
    run_dir.mkdir(parents=True, exist_ok=True)

    log_root = REPO_DIR / 'log' / dataset / 'SGL'
    before_logs = {p for p in log_root.glob('*.log')} if log_root.exists() else set()
    cmd = [sys.executable, 'main_phase3.py', *build_sgl_cli_args(dataset, cfg)]
    env = os.environ.copy()
    env['SGL_RUN_OUTPUT_DIR'] = run_dir.resolve().as_posix()

    print('=' * 90)
    print(f'Running official SGL repo: dataset={dataset}, {run_name}, seed={FIXED_SEED}')
    print('=' * 90)
    start = time.time()
    proc = subprocess.run(cmd, cwd=REPO_DIR, text=True, capture_output=True, env=env)
    elapsed = round(time.time() - start, 2)
    (run_dir / 'launcher.log').write_text(proc.stdout + '\n' + proc.stderr, encoding='utf-8')
    if proc.returncode != 0:
        raise RuntimeError(f'SGL failed for dataset={dataset}, {run_name}. Check {run_dir / "launcher.log"}.')

    repo_log_path = find_repo_log(log_root, before_logs, start)
    log_text = repo_log_path.read_text(encoding='utf-8')
    (run_dir / 'train.log').write_text(log_text, encoding='utf-8')

    history_rows, metric_names, best_epoch, stop_epoch = parse_metric_history(log_text)
    history_df = pd.DataFrame(history_rows)
    history_df.to_csv(run_dir / 'history.csv', index=False)

    training_summary_path = run_dir / 'training_summary.json'
    training_summary = json.loads(training_summary_path.read_text(encoding='utf-8')) if training_summary_path.exists() else {}
    extra_metrics = evaluate_extra_metrics(dataset, cfg, run_dir)

    final_best_epoch = best_epoch if best_epoch is not None else training_summary.get('best_epoch')
    final_stop_epoch = stop_epoch if stop_epoch is not None else training_summary.get('stop_epoch', cfg['epochs'])
    stopped_early = bool(final_stop_epoch is not None and int(final_stop_epoch) < int(cfg['epochs']))

    result = {
        'dataset': dataset,
        'model': 'SGL',
        'run_id': run_id,
        'run_name': run_name,
        'seed': FIXED_SEED,
        'best_eval_epoch': final_best_epoch,
        'stop_eval_epoch': final_stop_epoch,
        'stopped_early': stopped_early,
        'elapsed_seconds': elapsed,
        'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu',
        **extra_metrics,
    }
    (run_dir / 'metrics.json').write_text(json.dumps(result, indent=2), encoding='utf-8')
    return result


## Run all Phase 3 Part 2 experiments
This runs **5 datasets x 3 repeated runs with fixed seed 42 = 15 runs** using the official SGL repo.


In [ ]:
all_results = []
for dataset in DATASETS:
    for run_id in RUN_IDS:
        all_results.append(run_one(dataset, run_id))

results_df = pd.DataFrame(all_results).sort_values(['dataset', 'run_id']).reset_index(drop=True)
results_df.to_csv(RESULTS_DIR / 'all_runs.csv', index=False)
display(results_df)


In [ ]:
metric_cols = [c for c in results_df.columns if '@' in c] + ['elapsed_seconds']
summary_rows = []
for dataset, group in results_df.groupby('dataset', sort=True):
    row = {'dataset': dataset, 'model': 'SGL', 'runs': int(len(group))}
    for col in metric_cols:
        row[f'{col}_mean'] = float(group[col].mean())
        row[f'{col}_std'] = float(group[col].std(ddof=0))
    summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows).sort_values('dataset').reset_index(drop=True)
summary_df.to_csv(RESULTS_DIR / 'summary.csv', index=False)
summary_df.to_csv(AVG_RESULTS_DIR / 'summary_with_std.csv', index=False)

incomplete = summary_df.loc[summary_df['runs'] != len(RUN_IDS), 'dataset'].tolist()
if incomplete:
    raise RuntimeError(f'Cannot build 3-run averages. Incomplete datasets: {incomplete}')

average_report_df = pd.DataFrame({
    'Dataset': summary_df['dataset'],
    'Model': summary_df['model'],
    f'Recall@{REPORT_TOPK}': summary_df[f'recall@{REPORT_TOPK}_mean'].round(6),
    f'NDCG@{REPORT_TOPK}': summary_df[f'ndcg@{REPORT_TOPK}_mean'].round(6),
    f'HR@{REPORT_TOPK}': summary_df[f'hr@{REPORT_TOPK}_mean'].round(6),
    f'MRR@{REPORT_TOPK}': summary_df[f'mrr@{REPORT_TOPK}_mean'].round(6),
})
average_report_df.to_csv(AVG_RESULTS_DIR / 'average_metrics_3_runs.csv', index=False)
(AVG_RESULTS_DIR / 'average_metrics_3_runs.json').write_text(
    json.dumps(average_report_df.to_dict(orient='records'), indent=2),
    encoding='utf-8'
)

for row in summary_df.to_dict(orient='records'):
    dataset_payload = {
        'dataset': row['dataset'],
        'model': row['model'],
        'runs': row['runs'],
        'mean': {
            f'recall@{REPORT_TOPK}': row[f'recall@{REPORT_TOPK}_mean'],
            f'ndcg@{REPORT_TOPK}': row[f'ndcg@{REPORT_TOPK}_mean'],
            f'hr@{REPORT_TOPK}': row[f'hr@{REPORT_TOPK}_mean'],
            f'mrr@{REPORT_TOPK}': row[f'mrr@{REPORT_TOPK}_mean'],
            'elapsed_seconds': row['elapsed_seconds_mean'],
        },
        'std': {
            f'recall@{REPORT_TOPK}': row[f'recall@{REPORT_TOPK}_std'],
            f'ndcg@{REPORT_TOPK}': row[f'ndcg@{REPORT_TOPK}_std'],
            f'hr@{REPORT_TOPK}': row[f'hr@{REPORT_TOPK}_std'],
            f'mrr@{REPORT_TOPK}': row[f'mrr@{REPORT_TOPK}_std'],
            'elapsed_seconds': row['elapsed_seconds_std'],
        },
    }
    (AVG_RESULTS_DIR / f"{row['dataset']}_average.json").write_text(
        json.dumps(dataset_payload, indent=2),
        encoding='utf-8'
    )

    dataset_avg_dir = RESULTS_DIR / row['dataset'] / 'average_results_3_runs'
    dataset_avg_dir.mkdir(parents=True, exist_ok=True)
    dataset_avg_row_df = pd.DataFrame([
        {
            'Dataset': row['dataset'],
            'Model': row['model'],
            'Runs': row['runs'],
            f'Recall@{REPORT_TOPK}_mean': round(row[f'recall@{REPORT_TOPK}_mean'], 6),
            f'Recall@{REPORT_TOPK}_std': round(row[f'recall@{REPORT_TOPK}_std'], 6),
            f'NDCG@{REPORT_TOPK}_mean': round(row[f'ndcg@{REPORT_TOPK}_mean'], 6),
            f'NDCG@{REPORT_TOPK}_std': round(row[f'ndcg@{REPORT_TOPK}_std'], 6),
            f'HR@{REPORT_TOPK}_mean': round(row[f'hr@{REPORT_TOPK}_mean'], 6),
            f'HR@{REPORT_TOPK}_std': round(row[f'hr@{REPORT_TOPK}_std'], 6),
            f'MRR@{REPORT_TOPK}_mean': round(row[f'mrr@{REPORT_TOPK}_mean'], 6),
            f'MRR@{REPORT_TOPK}_std': round(row[f'mrr@{REPORT_TOPK}_std'], 6),
            'elapsed_seconds_mean': round(row['elapsed_seconds_mean'], 6),
            'elapsed_seconds_std': round(row['elapsed_seconds_std'], 6),
        }
    ])
    dataset_avg_row_df.to_csv(dataset_avg_dir / 'average_metrics_3_runs.csv', index=False)
    (dataset_avg_dir / 'average_metrics_3_runs.json').write_text(
        json.dumps(dataset_payload, indent=2),
        encoding='utf-8'
    )

display(summary_df)
display(average_report_df)
print(f'Global 3-run average reports saved to: {AVG_RESULTS_DIR}')
print(f'Per-dataset average folders saved under: {RESULTS_DIR / DATASETS[0] / "average_results_3_runs"} ...')


In [ ]:
def read_sgl_split_stats(dataset: str):
    folder = REPO_DIR / 'dataset' / dataset
    train_df = pd.read_csv(folder / f'{dataset}.train', header=None, names=['user', 'item'])
    test_df = pd.read_csv(folder / f'{dataset}.test', header=None, names=['user', 'item'])

    train_users = set(train_df['user'].astype(int).tolist())
    test_users = set(test_df['user'].astype(int).tolist())
    train_items = set(train_df['item'].astype(int).tolist())
    test_items = set(test_df['item'].astype(int).tolist())
    all_users = train_users | test_users
    all_items = train_items | test_items
    total_interactions = len(train_df) + len(test_df)
    density = total_interactions / (len(all_users) * len(all_items)) if all_users and all_items else 0.0

    return {
        'dataset': dataset,
        'users': len(all_users),
        'items': len(all_items),
        'train_interactions': int(len(train_df)),
        'test_interactions': int(len(test_df)),
        'total_interactions': int(total_interactions),
        'density': density,
    }


dataset_stats_df = pd.DataFrame(
    [read_sgl_split_stats(dataset) for dataset in DATASETS]
).sort_values('dataset').reset_index(drop=True)
dataset_stats_df.to_csv(RESULTS_DIR / 'dataset_characteristics.csv', index=False)

analysis_df = summary_df.merge(dataset_stats_df, on='dataset', how='left')
analysis_df.to_csv(RESULTS_DIR / 'sgl_phase3_part2_analysis.csv', index=False)

display(dataset_stats_df)

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
plot_specs = [
    (f'recall@{REPORT_TOPK}_mean', f'Recall@{REPORT_TOPK}', '#3366cc'),
    (f'ndcg@{REPORT_TOPK}_mean', f'NDCG@{REPORT_TOPK}', '#dc3912'),
    (f'hr@{REPORT_TOPK}_mean', f'HR@{REPORT_TOPK}', '#009688'),
    (f'mrr@{REPORT_TOPK}_mean', f'MRR@{REPORT_TOPK}', '#8e24aa'),
    ('elapsed_seconds_mean', 'Runtime (s)', '#ff9900'),
    ('density', 'Density', '#109618'),
]

for ax, (col, title, color) in zip(axes.flatten(), plot_specs):
    bars = ax.bar(analysis_df['dataset'], analysis_df[col], color=color)
    ax.set_title(title, fontweight='bold')
    ax.tick_params(axis='x', rotation=20)
    for bar, value in zip(bars, analysis_df[col]):
        label = f'{value:.4f}' if value < 1000 else f'{value:,.0f}'
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height(), label, ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.savefig(FIG_DIR / 'analysis_overview.png', dpi=150, bbox_inches='tight')
plt.show()

corr_cols = [
    'users', 'items', 'total_interactions', 'density',
    f'recall@{REPORT_TOPK}_mean', f'ndcg@{REPORT_TOPK}_mean',
    f'hr@{REPORT_TOPK}_mean', f'mrr@{REPORT_TOPK}_mean'
]
print('Correlation summary:')
display(analysis_df[corr_cols].corr(numeric_only=True))


In [ ]:
for history_path in sorted(RESULTS_DIR.glob('*/*/history.csv')):
    history_df = pd.read_csv(history_path)
    if history_df.empty:
        continue
    dataset = history_path.parent.parent.name
    run_name = history_path.parent.name
    plt.figure(figsize=(7, 4))
    for col in ['recall@20', 'ndcg@20', 'mrr@20']:
        if col in history_df.columns:
            plt.plot(history_df['eval_epoch'], history_df[col], label=col)
    plt.xlabel('Evaluation Epoch')
    plt.title(f'{dataset} - {run_name}')
    plt.legend()
    plt.tight_layout()
    plt.savefig(FIG_DIR / f'{dataset}_{run_name}_convergence.png', dpi=150, bbox_inches='tight')
    plt.close()

run_metadata = {
    'repo_url': REPO_URL,
    'datasets': DATASETS,
    'fixed_seed': FIXED_SEED,
    'run_ids': RUN_IDS,
    'topks': TOPKS,
    'report_topk': REPORT_TOPK,
    'metric_list': METRIC_LIST,
    'sgl_run_config': SGL_RUN_CONFIG,
    'torch_version': torch.__version__,
    'cuda_available': bool(torch.cuda.is_available()),
    'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu',
    'lastfm_input_dir': str(LASTFM_PHASE2_INPUT_DIR),
    'average_results_dir': str(AVG_RESULTS_DIR),
}
(RESULTS_DIR / 'run_metadata.json').write_text(json.dumps(run_metadata, indent=2), encoding='utf-8')

archive_base = ROOT / 'phase3_part2_sgl_results'
archive_zip = archive_base.with_suffix('.zip')
if archive_zip.exists():
    archive_zip.unlink()
shutil.make_archive(str(archive_base), 'zip', RESULTS_DIR)
print(f'Results archive: {archive_zip}')
print('Finished Phase 3 Part 2.')
